In [6]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("large-session")
    .master("spark://172.28.1.2:7077")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.executor.cores", 2)
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.dynamicAllocation.maxExecutors", 5)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.warehouse.dir", "hdfs://master/user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)


# spark_2 = SparkSession.builder \
#     .appName("airline_analytics") \
#     .master("spark://172.28.1.2:7077") \
#     .config("spark.driver.memory", "8g") \
#     .config("spark.driver.maxResultSize", "4g") \
#     .config("spark.executor.memory", "8g") \
#     .config("spark.executor.cores", "3") \
#     .config("spark.executor.instances", "2") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.4") \
#     .config("spark.sql.shuffle.partitions", "100") \
#     .config("spark.sql.files.maxPartitionBytes", "134217728") \
#     .config("spark.sql.adaptive.enabled", "true") \
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
#     .config("spark.sql.adaptive.skewJoin.enabled", "true") \
#     .config("spark.sql.warehouse.dir", "hdfs://master/user/hive/warehouse") \
#     .config("hive.metastore.uris", "thrift://master:9083") \
#     .enableHiveSupport() \
#     .getOrCreate()


In [7]:
df_ml = spark.sql("""
    SELECT
        DAYOFWEEK(TO_DATE(FL_DATE, 'yyyy-MM-dd'))   AS day_of_week,
        MONTH(TO_DATE(FL_DATE, 'yyyy-MM-dd'))        AS month,
        CAST(SUBSTRING(CRS_DEP_TIME, 1, 2) AS INT)  AS sched_dep_hour,
        CAST(SUBSTRING(CRS_ARR_TIME, 1, 2) AS INT)  AS sched_arr_hour,
        OP_CARRIER,
        ORIGIN,
        DEST,
        DEP_DELAY,
        ARR_DELAY
    FROM flights_clean
""")

df_ml.printSchema()
df_ml.show(5)

26/02/28 18:17:28 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/02/28 18:17:28 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/02/28 18:17:28 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/02/28 18:17:28 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


root
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- sched_dep_hour: integer (nullable = true)
 |-- sched_arr_hour: integer (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEP_DELAY: float (nullable = true)
 |-- ARR_DELAY: float (nullable = true)



+-----------+-----+--------------+--------------+----------+------+----+---------+---------+
|day_of_week|month|sched_dep_hour|sched_arr_hour|OP_CARRIER|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|
+-----------+-----+--------------+--------------+----------+------+----+---------+---------+
|          4|    7|            18|            22|        OO|   DLH| ORD|     25.0|    397.0|
|          4|    7|            20|            -2|        OO|   DEN| EAR|     22.0|    305.0|
|          4|    7|            16|            54|        OO|   DEN| RKS|     23.0|    260.0|
|          4|    7|            16|            -1|        OO|   CGI| PAH|     11.0|     45.0|
|          4|    7|            14|            12|        OO|   ORD| CGI|     36.0|    339.0|
+-----------+-----+--------------+--------------+----------+------+----+---------+---------+
only showing top 5 rows



In [27]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = full_data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in full_data.columns
])

null_counts.toPandas()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_TIME,ARR_DELAY
0,0,0,0,0,0,394022,1329823,1339059,422571,1419584,1121351


In [8]:
# Step 1 — Drop columns that are too sparse to use
df_ml = df_ml.drop("DEP_TIME", "ARR_TIME")
print("Columns after dropping sparse columns:")
print(df_ml.columns)

Columns after dropping sparse columns:
['day_of_week', 'month', 'sched_dep_hour', 'sched_arr_hour', 'OP_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY', 'ARR_DELAY']


In [9]:
# Step 2 — Drop rows where label or key features are null
df_ml = df_ml.dropna(
    subset=["ARR_DELAY", 
            "DEP_DELAY", 
            "sched_dep_hour", 
            "sched_arr_hour"])

print(f"Rows remaining after dropping nulls: {df_ml.count():,}")

Rows remaining after dropping nulls: 69,978,065


In [14]:
# Step 3 — Verify no nulls remain
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_ml.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_ml.columns
])

null_counts.toPandas()

,day_of_week,month,sched_dep_hour,sched_arr_hour,OP_CARRIER,ORIGIN,DEST,DEP_DELAY,ARR_DELAY
0,0,0,0,0,0,0,0,0,0


In [10]:
# Build pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline

carrier_indexer = StringIndexer(inputCol="OP_CARRIER", outputCol="carrier_idx", handleInvalid="skip")
origin_indexer  = StringIndexer(inputCol="ORIGIN",     outputCol="origin_idx",  handleInvalid="skip")
dest_indexer    = StringIndexer(inputCol="DEST",       outputCol="dest_idx",    handleInvalid="skip")

encoder = OneHotEncoder(
    inputCols=["carrier_idx", "origin_idx", "dest_idx"],
    outputCols=["carrier_vec", "origin_vec", "dest_vec"]
)

assembler = VectorAssembler(
    inputCols=[
        "DEP_DELAY",
        "sched_dep_hour",
        "sched_arr_hour",
        "day_of_week",
        "month",
        "carrier_vec",
        "origin_vec",
        "dest_vec"
    ],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="ARR_DELAY",
    maxIter=10,
    regParam=0.1,
    elasticNetParam=0
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="ARR_DELAY",
    numTrees=20,
    maxDepth=4,
    seed=42
)

In [11]:
# Train / Test split
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows : {train_df.count():,}")
print(f"Testing rows  : {test_df.count():,}")

Training rows : 55,979,489


Testing rows  : 13,998,576


In [12]:
# Train Linear Regression
lr_pipeline = Pipeline(stages=[
    carrier_indexer, origin_indexer, dest_indexer,
    encoder, assembler, lr
])

print("Training Linear Regression...")
lr_model = lr_pipeline.fit(train_df)
print("Done.")

Training Linear Regression...


26/02/28 18:22:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/28 18:22:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Done.


In [13]:
# Train Random Forest
rf_pipeline = Pipeline(stages=[
    carrier_indexer, origin_indexer, dest_indexer,
    encoder, assembler, rf
])

print("Training Random Forest...")
rf_model = rf_pipeline.fit(train_df)
print("Done.")

Training Random Forest...


26/02/28 18:28:29 WARN TaskSetManager: Lost task 8.0 in stage 36.0 (TID 251) (worker1 executor 4): java.io.IOException: No space left on device
	at sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at sun.nio.ch.FileDispatcherImpl.write(FileDispatcherImpl.java:60)
	at sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:93)
	at sun.nio.ch.IOUtil.write(IOUtil.java:65)
	at sun.nio.ch.FileChannelImpl.write(FileChannelImpl.java:211)
	at org.apache.spark.storage.CountingWritableChannel.write(DiskStore.scala:355)
	at java.nio.channels.Channels.writeFullyImpl(Channels.java:78)
	at java.nio.channels.Channels.writeFully(Channels.java:101)
	at java.nio.channels.Channels.access$000(Channels.java:61)
	at java.nio.channels.Channels$1.write(Channels.java:174)
	at java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:82)
	at java.io.BufferedOutputStream.write(BufferedOutputStream.java:126)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net

KeyboardInterrupt: 

26/02/28 18:32:18 WARN LeaseRenewer: Failed to renew lease for [DFSClient_NONMAPREDUCE_-1494472203_15] for 239 seconds.  Will retry shortly ...
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.hdfs.server.namenode.SafeModeException): Cannot renew lease for DFSClient_NONMAPREDUCE_-1494472203_15. Name node is in safe mode.
Resources are low on NN. Please add or free up more resourcesthen turn off safe mode manually. NOTE:  If you turn off safe mode before adding resources, the NN will immediately return to safe mode. Use "hdfs dfsadmin -safemode leave" to turn safe mode off. NamenodeHostName:master
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.newSafemodeException(FSNamesystem.java:1612)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.checkNameNodeSafeMode(FSNamesystem.java:1599)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.renewLease(FSNamesystem.java:4055)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.renewLease(NameNodeRpcServer.java:

In [ ]:
# Evaluate
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

evaluator_rmse = RegressionEvaluator(labelCol="ARR_DELAY", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="ARR_DELAY", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="ARR_DELAY", predictionCol="prediction", metricName="r2")

lr_preds = lr_model.transform(test_df)
rf_preds = rf_model.transform(test_df)

pd.DataFrame({
    "Metric"           : ["RMSE (mins)", "MAE (mins)", "R²"],
    "Linear Regression": [
        round(evaluator_rmse.evaluate(lr_preds), 4),
        round(evaluator_mae.evaluate(lr_preds),  4),
        round(evaluator_r2.evaluate(lr_preds),   4)
    ],
    "Random Forest"    : [
        round(evaluator_rmse.evaluate(rf_preds), 4),
        round(evaluator_mae.evaluate(rf_preds),  4),
        round(evaluator_r2.evaluate(rf_preds),   4)
    ]
})

26/02/28 11:45:15 ERROR TaskSchedulerImpl: Lost executor 8 on worker1: Command exited with code 137
26/02/28 11:45:15 WARN TaskSetManager: Lost task 8.1 in stage 39.0 (TID 227) (worker1 executor 8): ExecutorLostFailure (executor 8 exited caused by one of the running tasks) Reason: Command exited with code 137
26/02/28 11:45:15 WARN TaskSetManager: Lost task 4.1 in stage 39.0 (TID 226) (worker1 executor 8): ExecutorLostFailure (executor 8 exited caused by one of the running tasks) Reason: Command exited with code 137
26/02/28 11:45:15 WARN TaskSetManager: Lost task 0.1 in stage 39.0 (TID 228) (worker1 executor 8): ExecutorLostFailure (executor 8 exited caused by one of the running tasks) Reason: Command exited with code 137


In [14]:
try:
    spark.stop()
except:
    pass

26/02/28 18:32:51 ERROR Instrumentation: org.apache.spark.SparkException: Job 24 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1253)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1251)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1251)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3087)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:84)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:2973)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1375)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:2973)
	at org.apache.spark.SparkContext.$anonfun$stop$12(SparkContext.scala:2258)
	at org.apache.spark.util.Utils$.tryL

In [4]:
spark.sql("DESCRIBE flights_clean").show(truncate=False)

26/02/28 11:29:51 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/02/28 11:29:51 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|FL_DATE          |string   |NULL   |
|OP_CARRIER       |string   |NULL   |
|OP_CARRIER_FL_NUM|string   |NULL   |
|ORIGIN           |string   |NULL   |
|DEST             |string   |NULL   |
|CRS_DEP_TIME     |string   |NULL   |
|DEP_TIME         |string   |NULL   |
|DEP_DELAY        |float    |NULL   |
|CRS_ARR_TIME     |string   |NULL   |
|ARR_TIME         |string   |NULL   |
|ARR_DELAY        |float    |NULL   |
+-----------------+---------+-------+

